# Week 6 Lab: Monte Carlo Simulation
20232019 Maksutova Aibike

In [ ]:
import random
import math
import matplotlib.pyplot as plt

## 실습 1. LCG 구현 and histogram of random numbers in [0, 99]

In [ ]:
class LCG:
    def __init__(self, seed=1, a=1664525, c=1013904223, m=2**32):
        self.a, self.c, self.m = a, c, m
        self.x = seed

    def next(self):
        # X(n+1) = (a * X(n) + c) mod m
        self.x = (self.a * self.x + self.c) % self.m
        return self.x

    def randint_0_99(self):
        # scale the number from [0, m) to [0, 99]
        return self.next() * 100 // self.m

lcg = LCG(seed=42)
counts = [100, 1000, 10000, 100000]

plt.figure(figsize=(12, 8))
for k, n in enumerate(counts):
    nums = [lcg.randint_0_99() for _ in range(n)]
    plt.subplot(2, 2, k + 1)
    plt.hist(nums, bins=100, range=(0, 100))
    plt.title(f"LCG, {n} numbers")
plt.tight_layout()
plt.show()

When we generate more numbers, the histogram becomes flatter, so the distribution is close to uniform.

## 실습 2. LCG vs Mersenne Twister (Python random)

In [ ]:
N = 100000
lcg = LCG(seed=42)
lcg_nums = [lcg.randint_0_99() for _ in range(N)]
mt_nums = [random.randint(0, 99) for _ in range(N)]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(lcg_nums, bins=100, range=(0, 100))
plt.title("LCG")
plt.subplot(1, 2, 2)
plt.hist(mt_nums, bins=100, range=(0, 100), color="orange")
plt.title("Mersenne Twister")
plt.tight_layout()
plt.show()

# how far each histogram is from perfectly uniform (smaller = more uniform)
def deviation(nums):
    expected = len(nums) / 100
    cnt = [0] * 100
    for v in nums:
        cnt[v] += 1
    return sum((c - expected) ** 2 / expected for c in cnt)   # chi-square value

print("LCG chi-square:", round(deviation(lcg_nums), 2))
print("MT  chi-square:", round(deviation(mt_nums), 2))

## 실습 3. Dice: probability that the sum of two dice is 8

In [ ]:
def dice_sum_8(trials):
    hit = 0
    for _ in range(trials):
        d1 = random.randint(1, 6)
        d2 = random.randint(1, 6)
        if d1 + d2 == 8:
            hit += 1
    return hit / trials

real = 5 / 36   # (2,6) (3,5) (4,4) (5,3) (6,2)
for trials in [100, 1000, 10000, 100000, 1000000]:
    p = dice_sum_8(trials)
    err = abs(p - real) / real * 100
    print(f"trials = {trials:>7}   estimate = {p:.5f}   real = {real:.5f}   error = {err:.3f}%")

## 코딩 과제 6. Compute π with Monte Carlo

In [ ]:
MAX_POINTS = 1000000

inside = 0
in_x, in_y, out_x, out_y = [], [], [], []
DRAW = 5000   # draw only the first 5000 points, drawing 1,000,000 points is too slow

for i in range(MAX_POINTS):
    x = random.uniform(-1, 1)
    y = random.uniform(-1, 1)
    if x * x + y * y <= 1:        # inside the circle of radius 1
        inside += 1
        if i < DRAW:
            in_x.append(x); in_y.append(y)
    else:
        if i < DRAW:
            out_x.append(x); out_y.append(y)

# (circle area) / (square area) = pi / 4
pi_est = 4 * inside / MAX_POINTS
err = abs(pi_est - math.pi) / math.pi * 100
print("estimated pi =", pi_est)
print("real pi      =", math.pi)
print(f"error        = {err:.4f}%")

plt.figure(figsize=(6, 6))
plt.scatter(in_x, in_y, s=1, color="red", label="inside")
plt.scatter(out_x, out_y, s=1, color="blue", label="outside")
plt.gca().set_aspect("equal")
plt.title(f"Monte Carlo pi = {pi_est:.5f}")
plt.legend(loc="upper right")
plt.show()

With 1,000,000 points the error is usually below 0.1%. If it is not, run again or increase MAX_POINTS.